In [ ]:
# ============================================================
#  环境配置
#  - Colab 用户：取消注释下方 Colab 区块
#  - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install "torch>=2.1" "transformers>=4.37" matplotlib pillow requests -q

# ── 本地 Jupyter 环境 ──
import importlib
import subprocess
import sys


def _install(requirement, import_name=None):
    name = import_name or requirement
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", requirement, "-q"])


for requirement, import_name in [
    ("torch>=2.1", "torch"),
    ("transformers>=4.37", "transformers"),
    ("matplotlib", "matplotlib"),
    ("pillow", "PIL"),
    ("requests", "requests"),
]:
    _install(requirement, import_name)

In [ ]:
import math
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, GroupViTModel

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# GroupViT 实战：架构解析 + 推理应用

基于论文 *GroupViT: Semantic Segmentation Emerges from Text Supervision* (Xu et al., 2022)，
围绕 **zero-shot semantic segmentation** 展示 GroupViT 的核心分组机制与预训练推理流程。

> 实现策略：**C — Application / Inference**  
> 选择理由：GroupViT 依赖大规模图文对预训练，CPU 上无法在教学场景中从零预训练；更适合“先拆架构，再调用预训练模型做 zero-shot 分割”。

| | 架构解析 | 推理应用 |
|---|---|---|
| 核心思路 | 逐组件拆解 Patch Embedding、Group Tokens、Grouping Block、层级分组与对比学习目标 | 加载 `nvidia/groupvit-gcc-yfcc` 预训练模型，完成 zero-shot 分类与语义分割 |
| 学习重点 | 为什么文本监督也能“长出”分割区域 | 如何准备 prompts、如何读取 `segmentation_logits`、如何解释结果 |
| 适合场景 | 面试准备、原理复盘 | 快速上手、实验验证 |
| 已知限制 | toy 代码只做机制演示，不复现原论文训练规模 | 依赖网络下载权重与示例图片，且候选类别设计会显著影响结果 |

## 1. 数据准备：选择最小可运行演示样本

本 Notebook 不做 GroupViT 预训练，而是选择 **单张公开图像 + 一组候选文本标签** 来演示模型的核心行为：

1. **图像侧**：准备一张包含多个目标的自然场景图片；
2. **文本侧**：准备分类 prompt 和分割 labels；
3. **目标**：观察 GroupViT 如何把图像区域与文本语义对齐。

这种设置满足教学场景的三个要求：
- 免费 Colab / 本地 CPU 都能运行；
- 不需要手工下载复杂数据集；
- 能直接看到 zero-shot 分类与分割结果。

In [ ]:
MODEL_ID = "nvidia/groupvit-gcc-yfcc"
DEMO_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"

CLASS_PROMPTS_FINE = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo of a remote control",
    "a photo of a couch",
    "a photo of a blanket",
]
CLASS_PROMPTS_COARSE = [
    "an animal",
    "a pet",
    "electronics",
    "furniture",
    "fabric",
]
SEGMENT_LABELS = ["cat", "remote control", "couch", "blanket"]


def load_image_from_url(url: str) -> Image.Image:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")


image = load_image_from_url(DEMO_URL)
print(f"Model ID: {MODEL_ID}")
print(f"Image size: {image.size}, mode: {image.mode}")
print(f"Fine prompts: {len(CLASS_PROMPTS_FINE)}")
print(f"Segmentation labels: {SEGMENT_LABELS}")

In [ ]:
PREVIEW_SIZE = 224
PATCH_SIZE = 16

image_resized = image.resize((PREVIEW_SIZE, PREVIEW_SIZE))
image_array = np.asarray(image_resized, dtype=np.float32) / 255.0
image_tensor = torch.from_numpy(image_array).permute(2, 0, 1).unsqueeze(0)  # (1, 3, 224, 224)
patch_grid = PREVIEW_SIZE // PATCH_SIZE
num_patches = patch_grid * patch_grid

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image)
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(image_tensor[0].permute(1, 2, 0).numpy())
axes[1].set_title("Resized image")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"Tensor shape: {tuple(image_tensor.shape)}")
print(f"Patch grid: {patch_grid} x {patch_grid}")
print(f"Total patches: {num_patches}")

## 2. 共享设置：统一 toy 架构的超参数

下面的超参数只服务于 **机制演示版 GroupViT 编码器**，目标是：
- 尽量贴近论文的数据流；
- 尽量缩小规模，保证 CPU 上可立即运行；
- 让训练路径和推理路径都能独立展示。

In [ ]:
IMG_SIZE = 224
PATCH_SIZE = 16
PATCH_GRID = IMG_SIZE // PATCH_SIZE
NUM_PATCHES = PATCH_GRID * PATCH_GRID
D_MODEL = 64
NUM_HEADS = 4
NUM_GROUP_TOKENS_STAGE1 = 64
NUM_GROUP_TOKENS_STAGE2 = 8
NUM_ENCODER_LAYERS_STAGE1 = 2
NUM_ENCODER_LAYERS_STAGE2 = 1
BATCH_SIZE = 2
TEMPERATURE = 0.07

print(f"IMG_SIZE={IMG_SIZE}, PATCH_GRID={PATCH_GRID}, NUM_PATCHES={NUM_PATCHES}")
print(f"D_MODEL={D_MODEL}, heads={NUM_HEADS}")
print(f"Group tokens: {NUM_GROUP_TOKENS_STAGE1} -> {NUM_GROUP_TOKENS_STAGE2}")

## 3. 架构解析：GroupViT 为什么能从文本监督中“长出”分割区域？

这一节先不调用预训练模型，而是手写一个 **toy GroupViT encoder**，把论文里最关键的 5 个动作串起来：

1. patch embedding；
2. group tokens；
3. grouping block；
4. hierarchical grouping；
5. contrastive objective。

重点不是复现实验指标，而是把 **张量形状、公式、训练路径、推理路径** 全部走清楚。

### 3.1 Patch Embedding

GroupViT 的输入仍然是 ViT 风格的 patch 序列。对于输入图像 $\mathbf{X} \in \mathbb{R}^{B \times 3 \times H \times W}$，先把图像切成 patch，再映射到统一维度：

$$
\mathbf{P} = \text{Proj}(\text{Patchify}(\mathbf{X})) \in \mathbb{R}^{B \times M \times D}
$$

其中：
- $M = (H / P) \times (W / P)$ 是 patch 数量；
- $D$ 是 token 维度。

**为什么这样做有效？**  
因为 Transformer 擅长处理“序列”，而 patch embedding 把二维图像变成了有顺序的 token 序列，后续的 attention 和 grouping 才有统一的输入表示。

- 输入形状：`(batch, 3, 224, 224)`
- 输出形状：`(batch, 196, d_model)`

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=64):
        super().__init__()
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, embed_dim) * 0.02)

    def forward(self, pixel_values):
        # (batch, 3, 224, 224) -> (batch, d_model, 14, 14)
        x = self.proj(pixel_values)
        # (batch, d_model, 14, 14) -> (batch, 196, d_model)
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        return x


patch_embed_demo = PatchEmbedding(img_size=IMG_SIZE, patch_size=PATCH_SIZE, embed_dim=D_MODEL).to(device)
dummy_pixels = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=device)
patch_tokens = patch_embed_demo(dummy_pixels)

print(f"Input shape: {tuple(dummy_pixels.shape)}")
print(f"Patch tokens shape: {tuple(patch_tokens.shape)}")

### 3.2 Group Tokens：把“聚类中心”显式参数化

普通分类 ViT 往往只需要一个全局 token；但分割任务中，一张图通常包含多个语义区域，因此 GroupViT 显式引入多个 **group tokens**：

$$
\mathbf{G}_1 \in \mathbb{R}^{1 \times K_1 \times D}, \qquad
\mathbf{G}_2 \in \mathbb{R}^{1 \times K_2 \times D}
$$

- 第一阶段：$K_1 = 64$ 个 group tokens；
- 第二阶段：$K_2 = 8$ 个 group tokens。

**直觉理解**：它们就像“可学习的聚类中心”，让 patch tokens 不再只能保留在规则网格里，而是可以逐步聚合成任意形状的语义区域。

- patch tokens 形状：`(batch, 196, d_model)`
- group tokens 形状：`(batch, 64, d_model)` 或 `(batch, 8, d_model)`

In [ ]:
group_tokens_stage1 = nn.Parameter(torch.randn(1, NUM_GROUP_TOKENS_STAGE1, D_MODEL, device=device) * 0.02)
group_tokens_stage2 = nn.Parameter(torch.randn(1, NUM_GROUP_TOKENS_STAGE2, D_MODEL, device=device) * 0.02)

expanded_stage1 = group_tokens_stage1.expand(BATCH_SIZE, -1, -1)  # (batch, 64, d_model)
combined_stage1 = torch.cat([patch_tokens, expanded_stage1], dim=1)  # (batch, 196 + 64, d_model)

print(f"Patch tokens: {tuple(patch_tokens.shape)}")
print(f"Stage1 group tokens: {tuple(expanded_stage1.shape)}")
print(f"Combined sequence: {tuple(combined_stage1.shape)}")

### 3.3 Grouping Block：把 patch 分配给 group

Grouping Block 是 GroupViT 的核心。它先计算 patch token 与 group token 的相似度，再把 patch 聚合到 group 上。

**相似度矩阵：**

$$
\mathbf{S} = \frac{\mathbf{P} \mathbf{G}^{\top}}{\sqrt{D}} \in \mathbb{R}^{B \times M \times K}
$$

**训练时的分配：**

$$
\mathbf{A}_{\text{train}} = \text{GumbelSoftmax}(\mathbf{S})
$$

**推理时的分配：**

$$
\mathbf{A}_{\text{infer}} = \text{OneHot}(\arg\max \mathbf{S})
$$

**聚合得到新的 group 表示：**

$$
\mathbf{G}' = \hat{\mathbf{A}}^{\top} \mathbf{P}
$$

其中 $\hat{\mathbf{A}}$ 是列归一化后的分配矩阵。

**为什么这样做有效？**  
因为模型不再被迫在固定网格里做分割，而是允许“谁和谁属于同一组”由相似度自适应决定。

- 输入：`image_tokens -> (batch, M, d_model)`，`group_tokens -> (batch, K, d_model)`
- 输出：`new_groups -> (batch, K, d_model)`，`assign_matrix -> (batch, M, K)`

In [ ]:
class GroupingBlock(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.norm_image = nn.LayerNorm(embed_dim)
        self.norm_group = nn.LayerNorm(embed_dim)
        self.scale = embed_dim ** -0.5
        self.refine = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Linear(embed_dim * 2, embed_dim),
        )
        self.out_norm = nn.LayerNorm(embed_dim)

    def forward(self, image_tokens, group_tokens, training=True):
        # image_tokens: (batch, M, d_model)
        # group_tokens: (batch, K, d_model)
        image_tokens = self.norm_image(image_tokens)
        group_tokens = self.norm_group(group_tokens)

        # (batch, M, d_model) x (batch, d_model, K) -> (batch, M, K)
        scores = torch.matmul(image_tokens, group_tokens.transpose(-1, -2)) * self.scale

        if training:
            assign_matrix = F.gumbel_softmax(scores, tau=1.0, hard=True, dim=-1)
        else:
            assign_matrix = F.one_hot(scores.argmax(dim=-1), num_classes=group_tokens.shape[1]).float()

        assign_norm = assign_matrix / (assign_matrix.sum(dim=1, keepdim=True) + 1e-6)
        # (batch, K, M) x (batch, M, d_model) -> (batch, K, d_model)
        new_groups = torch.matmul(assign_norm.transpose(1, 2), image_tokens)
        new_groups = new_groups + self.refine(self.out_norm(new_groups))
        return new_groups, assign_matrix

### 3.4 Hierarchical Grouping：从细粒度块到粗粒度语义区域

GroupViT 不只做一次 grouping，而是分两阶段：

| Stage | 输入 token | group token 数量 | 输出 |
|---|---|---:|---|
| Stage 1 | 196 个 patch tokens | 64 | 64 个中间 groups |
| Stage 2 | 64 个中间 groups | 8 | 8 个最终 groups |

这相当于：
- **第一层** 先把局部相似 patch 聚成小块；
- **第二层** 再把小块合并成更完整的语义区域。

这也是为什么论文里浅层更像颜色/纹理块，深层更像“猫、沙发、地毯”这样的语义对象。

In [ ]:
class ToyGroupViTEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size=IMG_SIZE, patch_size=PATCH_SIZE, embed_dim=D_MODEL)
        self.num_patches = self.patch_embed.num_patches

        self.group_tokens_stage1 = nn.Parameter(torch.randn(1, NUM_GROUP_TOKENS_STAGE1, D_MODEL) * 0.02)
        self.group_tokens_stage2 = nn.Parameter(torch.randn(1, NUM_GROUP_TOKENS_STAGE2, D_MODEL) * 0.02)

        encoder_layer_stage1 = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NUM_HEADS,
            dim_feedforward=D_MODEL * 2,
            dropout=0.0,
            batch_first=True,
        )
        self.stage1_encoder = nn.TransformerEncoder(encoder_layer_stage1, num_layers=NUM_ENCODER_LAYERS_STAGE1)
        self.grouping_stage1 = GroupingBlock(D_MODEL)

        encoder_layer_stage2 = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NUM_HEADS,
            dim_feedforward=D_MODEL * 2,
            dropout=0.0,
            batch_first=True,
        )
        self.stage2_encoder = nn.TransformerEncoder(encoder_layer_stage2, num_layers=NUM_ENCODER_LAYERS_STAGE2)
        self.grouping_stage2 = GroupingBlock(D_MODEL)
        self.final_norm = nn.LayerNorm(D_MODEL)

    def forward(self, pixel_values, return_aux=False):
        batch_size = pixel_values.shape[0]

        patch_tokens = self.patch_embed(pixel_values)  # (batch, 196, d_model)

        stage1_seed = self.group_tokens_stage1.expand(batch_size, -1, -1)  # (batch, 64, d_model)
        stage1_tokens = torch.cat([patch_tokens, stage1_seed], dim=1)  # (batch, 260, d_model)
        stage1_tokens = self.stage1_encoder(stage1_tokens)
        stage1_patch_tokens = stage1_tokens[:, :self.num_patches, :]  # (batch, 196, d_model)
        stage1_group_seed = stage1_tokens[:, self.num_patches:, :]  # (batch, 64, d_model)
        stage1_groups, stage1_assign = self.grouping_stage1(
            stage1_patch_tokens,
            stage1_group_seed,
            training=self.training,
        )

        stage2_seed = self.group_tokens_stage2.expand(batch_size, -1, -1)  # (batch, 8, d_model)
        stage2_tokens = torch.cat([stage1_groups, stage2_seed], dim=1)  # (batch, 72, d_model)
        stage2_tokens = self.stage2_encoder(stage2_tokens)
        stage2_input_groups = stage2_tokens[:, :NUM_GROUP_TOKENS_STAGE1, :]  # (batch, 64, d_model)
        stage2_group_seed = stage2_tokens[:, NUM_GROUP_TOKENS_STAGE1:, :]  # (batch, 8, d_model)
        final_groups, stage2_assign = self.grouping_stage2(
            stage2_input_groups,
            stage2_group_seed,
            training=self.training,
        )

        image_embed = self.final_norm(final_groups.mean(dim=1))  # (batch, d_model)

        if return_aux:
            aux = {
                "patch_tokens": patch_tokens,
                "stage1_groups": stage1_groups,
                "final_groups": final_groups,
                "stage1_assign": stage1_assign,
                "stage2_assign": stage2_assign,
            }
            return image_embed, aux

        return image_embed


toy_encoder = ToyGroupViTEncoder().to(device)
toy_encoder.eval()
with torch.inference_mode():
    toy_image_embeds, toy_aux = toy_encoder(dummy_pixels, return_aux=True)

print(f"Image embedding: {tuple(toy_image_embeds.shape)}")
print(f"Stage1 assignment: {tuple(toy_aux['stage1_assign'].shape)}")
print(f"Stage2 assignment: {tuple(toy_aux['stage2_assign'].shape)}")
print(f"Final groups: {tuple(toy_aux['final_groups'].shape)}")

### 3.5 对比学习目标：训练时并没有像素标签

GroupViT 训练时并不直接监督“每个像素属于哪一类”，而是像 CLIP 一样，用图像全局表示和文本表示做对比学习：

$$
\mathcal{L} = -\frac{1}{2} \left[
\text{CE}(\mathbf{Z}_I \mathbf{Z}_T^{\top} / \tau)
+ 
\text{CE}(\mathbf{Z}_T \mathbf{Z}_I^{\top} / \tau)
\right]
$$

这里：
- $\mathbf{Z}_I$ 是图像嵌入；
- $\mathbf{Z}_T$ 是文本嵌入；
- $\tau$ 是温度参数。

**关键点**：  
模型在训练时只知道“这张图和这段文本匹不匹配”，但由于图像编码器内部有 grouping 机制，它会逐渐学到能支撑文本对齐的语义区域。

In [ ]:
def contrastive_loss(image_embeds, text_embeds, temperature=0.07):
    image_embeds = F.normalize(image_embeds, dim=-1)
    text_embeds = F.normalize(text_embeds, dim=-1)
    logits = torch.matmul(image_embeds, text_embeds.T) / temperature  # (batch, batch)
    labels = torch.arange(logits.shape[0], device=logits.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return 0.5 * (loss_i2t + loss_t2i)


dummy_text_embeds = torch.randn(BATCH_SIZE, D_MODEL, device=device)
loss_demo = contrastive_loss(toy_image_embeds, dummy_text_embeds, temperature=TEMPERATURE)
print(f"Toy contrastive loss: {loss_demo.item():.4f}")

### 3.6 训练 vs 推理的区别

`training_vs_inference_diff = True`，因为 GroupViT 的两条路径目标完全不同：

| 阶段 | 行为 | 代码位置 |
|---|---|---|
| 训练 | 使用 `Gumbel-Softmax` 做近似离散分配；图像嵌入与文本嵌入做对比学习；不需要像素标签 | 下一格 `toy training step` |
| 推理 | 使用 `argmax` 做硬分配；将最终 group 与候选类别文本做相似度计算，再把类别回投到 patch / pixel | 下两格 `toy inference path` + `pretrained inference` |

下面先把训练路径单独跑通，再切换到推理路径。

In [ ]:
optimizer = torch.optim.AdamW(toy_encoder.parameters(), lr=1e-3)

dummy_pixels_train = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=device)
dummy_text_train = torch.randn(BATCH_SIZE, D_MODEL, device=device)

toy_encoder.train()
optimizer.zero_grad()
train_image_embeds, train_aux = toy_encoder(dummy_pixels_train, return_aux=True)
train_loss = contrastive_loss(train_image_embeds, dummy_text_train, temperature=TEMPERATURE)
train_loss.backward()
optimizer.step()

print(f"Train image embeds: {tuple(train_image_embeds.shape)}")
print(f"Train loss: {train_loss.item():.4f}")
print(f"Stage1 hard assignment row sum: {train_aux['stage1_assign'][0, 0].sum().item():.1f}")

### 3.7 推理路径：把最终 group 的类别再“回投”到 patch

推理时，模型已经有了最终的 group 表示。接下来要做两件事：

1. 用文本特征给每个 final group 选类别；
2. 利用两阶段 assignment，把 final group 的类别分数一路传回 patch。

如果记：
- `A1` 为 `patch -> stage1 group` 分配；
- `A2` 为 `stage1 group -> final group` 分配；
- `G_final` 为最终 group 特征；
- `T` 为文本类别特征；

那么 patch 级别的类别分数可以写成：

$$
\text{PatchLogits} = (\mathbf{A}_1 \mathbf{A}_2) (\mathbf{G}_{final} \mathbf{T}^{\top})
$$

这就是 GroupViT 能从文本监督“长出” zero-shot segmentation 的核心链路。

In [ ]:
def backproject_segmentation(final_groups, stage1_assign, stage2_assign, text_features, img_size, patch_size):
    # final_groups: (batch, 8, d_model)
    # stage1_assign: (batch, 196, 64)
    # stage2_assign: (batch, 64, 8)
    # text_features: (num_labels, d_model)
    final_groups = F.normalize(final_groups, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

    # (batch, 8, d_model) x (d_model, num_labels) -> (batch, 8, num_labels)
    final_group_scores = torch.matmul(final_groups, text_features.T)
    # (batch, 196, 64) x (batch, 64, 8) -> (batch, 196, 8)
    patch_to_final = torch.matmul(stage1_assign, stage2_assign)
    # (batch, 196, 8) x (batch, 8, num_labels) -> (batch, 196, num_labels)
    patch_logits = torch.matmul(patch_to_final, final_group_scores)

    grid_size = img_size // patch_size
    dense_logits = patch_logits.transpose(1, 2).reshape(final_groups.shape[0], text_features.shape[0], grid_size, grid_size)
    upsampled_logits = F.interpolate(dense_logits, size=(img_size, img_size), mode="bilinear", align_corners=False)
    return patch_logits, upsampled_logits


toy_encoder.eval()
with torch.inference_mode():
    eval_image_embed, eval_aux = toy_encoder(dummy_pixels[:1], return_aux=True)
    toy_text_features = torch.randn(len(SEGMENT_LABELS), D_MODEL, device=device)
    toy_patch_logits, toy_dense_logits = backproject_segmentation(
        eval_aux["final_groups"],
        eval_aux["stage1_assign"],
        eval_aux["stage2_assign"],
        toy_text_features,
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
    )

toy_patch_mask = toy_patch_logits.argmax(dim=-1)[0].reshape(PATCH_GRID, PATCH_GRID).detach().cpu().numpy()
plt.figure(figsize=(4, 4))
plt.imshow(toy_patch_mask, cmap="tab20")
plt.title("Toy patch prediction")
plt.axis("off")
plt.show()

print(f"Toy patch logits: {tuple(toy_patch_logits.shape)}")
print(f"Toy dense logits: {tuple(toy_dense_logits.shape)}")

## 4. 推理应用：调用官方预训练 GroupViT

根据 Hugging Face 官方文档，当前推荐流程是：

1. `AutoProcessor.from_pretrained(...)` 准备图像和文本；
2. `GroupViTModel.from_pretrained(...)` 加载权重；
3. 普通前向可读取 `logits_per_image` 做 zero-shot classification；
4. 设置 `output_segmentation=True` 可读取 `segmentation_logits` 做 zero-shot segmentation。

下面直接在同一张图像上演示两类任务。

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
groupvit_model = GroupViTModel.from_pretrained(MODEL_ID).to(device)
groupvit_model.eval()

print(f"Loaded model: {MODEL_ID}")
print(f"Projection dim: {groupvit_model.config.projection_dim}")
print(f"Image size in config: {groupvit_model.config.vision_config.image_size}")

In [ ]:
def run_zero_shot_classification(pil_image, prompts):
    inputs = processor(text=prompts, images=pil_image, padding=True, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = groupvit_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=-1)[0].detach().cpu()
    return probs

In [ ]:
fine_probs = run_zero_shot_classification(image, CLASS_PROMPTS_FINE)
coarse_probs = run_zero_shot_classification(image, CLASS_PROMPTS_COARSE)

fine_top_idx = fine_probs.argmax().item()
coarse_top_idx = coarse_probs.argmax().item()

print("Fine prompts top-1:")
print(f"  label={CLASS_PROMPTS_FINE[fine_top_idx]}, prob={fine_probs[fine_top_idx].item():.4f}")
print("Coarse prompts top-1:")
print(f"  label={CLASS_PROMPTS_COARSE[coarse_top_idx]}, prob={coarse_probs[coarse_top_idx].item():.4f}")

### 4.1 Zero-shot segmentation

这一步把文本 labels 当作分割类别词表。模型会返回每个类别对应的 `segmentation_logits`，再通过双线性插值恢复到原图大小。

**注意**：输出质量强依赖文本标签设计。
- 标签过粗：语义边界容易模糊；
- 标签过细：类别间竞争更强，容易互相挤占；
- 背景类通常最难，因为图文监督对背景概念天然较弱。

In [ ]:
def run_zero_shot_segmentation(pil_image, labels):
    inputs = processor(text=labels, images=pil_image, padding=True, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = groupvit_model(**inputs, output_segmentation=True)
    seg_logits = outputs.segmentation_logits.detach().cpu().float()
    seg_logits = F.interpolate(seg_logits, size=pil_image.size[::-1], mode="bilinear", align_corners=False)
    return seg_logits


seg_logits = run_zero_shot_segmentation(image, SEGMENT_LABELS)
seg_probs = seg_logits.softmax(dim=1)[0]  # (num_labels, H, W)

fig, axes = plt.subplots(1, len(SEGMENT_LABELS) + 1, figsize=(4 * (len(SEGMENT_LABELS) + 1), 4))
axes[0].imshow(image)
axes[0].set_title("Input image")
axes[0].axis("off")

for idx, label in enumerate(SEGMENT_LABELS):
    axes[idx + 1].imshow(seg_probs[idx].numpy(), cmap="magma")
    axes[idx + 1].set_title(label)
    axes[idx + 1].axis("off")

plt.tight_layout()
plt.show()

print(f"Segmentation logits shape: {tuple(seg_logits.shape)}")

In [ ]:
pred_mask = seg_probs.argmax(dim=0).numpy()
colors = plt.cm.Set2(np.linspace(0, 1, len(SEGMENT_LABELS)))
color_mask = np.zeros((pred_mask.shape[0], pred_mask.shape[1], 3), dtype=np.float32)
for idx, color in enumerate(colors):
    color_mask[pred_mask == idx] = color[:3]

seg_pixel_ratio = np.array([(pred_mask == idx).mean() for idx in range(len(SEGMENT_LABELS))])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(image)
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(color_mask)
axes[1].set_title("Predicted mask")
axes[1].axis("off")

axes[2].imshow(image)
axes[2].imshow(color_mask, alpha=0.45)
axes[2].set_title("Overlay")
axes[2].axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.bar(SEGMENT_LABELS, seg_pixel_ratio, color="seagreen")
plt.title("Predicted pixel ratio")
plt.ylabel("Ratio")
plt.xticks(rotation=20)
plt.show()

## 5. 结果对比：prompt 粒度会直接影响输出

为了更接近真实使用场景，这里同时比较：
- **细粒度 prompt**：`cat / remote control / couch / blanket` 这样的具体类别；
- **粗粒度 prompt**：`animal / furniture / fabric` 这样的抽象概念。

这不是严格 benchmark，而是帮助你建立一个工程直觉：
**GroupViT 的 zero-shot 结果不仅受图像内容影响，还显著受候选词表设计影响。**

In [ ]:
def shorten_prompt(prompt):
    return (
        prompt.replace("a photo of a ", "")
        .replace("an ", "")
        .replace("a ", "")
    )


fine_names = [shorten_prompt(p) for p in CLASS_PROMPTS_FINE]
coarse_names = [shorten_prompt(p) for p in CLASS_PROMPTS_COARSE]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].bar(fine_names, fine_probs.numpy(), color="steelblue")
axes[0].set_title("Fine prompt probabilities")
axes[0].set_ylabel("Probability")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(coarse_names, coarse_probs.numpy(), color="darkorange")
axes[1].set_title("Coarse prompt probabilities")
axes[1].set_ylabel("Probability")
axes[1].tick_params(axis="x", rotation=20)

axes[2].bar(SEGMENT_LABELS, seg_pixel_ratio, color="slateblue")
axes[2].set_title("Segmentation pixel ratio")
axes[2].set_ylabel("Ratio")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

print("Top-1 fine prompt:", fine_names[fine_top_idx])
print("Top-1 coarse prompt:", coarse_names[coarse_top_idx])

### 结果总结

**在这份 Notebook 中，你应重点记住 4 件事：**

1. **GroupViT 不是先有 mask 再学语义**，而是先通过图文对齐学到有区分度的 group，再在推理时把 group 回投到像素；
2. **Grouping Block 是关键创新**，它把“固定网格 token”变成了“可学习聚类中心 + 自适应区域聚合”；
3. **训练和推理是两条不同路径**：训练用对比学习 + Gumbel-Softmax，推理用文本类别相似度 + 回投；
4. **prompt engineering 很重要**：zero-shot 分割不只是模型问题，还是标签词表设计问题。

**适用条件**：
- 想理解 GroupViT 的数据流与核心机制；
- 想快速验证某个类别词表在一张图上的 zero-shot 行为；
- 不追求复现论文训练规模和 mIoU 指标。

**边界与局限**：
- toy 架构没有复现论文的大规模训练；
- 单图示例只能说明机制，不能代表整体 benchmark；
- 背景类与长尾类通常不稳定。

## Appendix A：可视化补充

最后再补两种最有价值的可视化：
1. patch positional embedding 的空间形状；
2. 第一阶段 assignment matrix 的局部切片。

它们分别帮助你理解：
- ViT 里的位置信息如何保留二维布局；
- Grouping Block 是否真的在做“聚类式分配”。

In [ ]:
pos_map = toy_encoder.patch_embed.pos_embed[0, :, 0].detach().cpu().reshape(PATCH_GRID, PATCH_GRID)
assign_preview = eval_aux["stage1_assign"][0].detach().cpu()[:64, :16].T

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(pos_map, cmap="coolwarm")
axes[0].set_title("Positional embedding channel 0")
axes[0].axis("off")

axes[1].imshow(assign_preview, aspect="auto", cmap="viridis")
axes[1].set_title("Stage1 assignment preview")
axes[1].set_xlabel("Patch index")
axes[1].set_ylabel("Group index")

plt.tight_layout()
plt.show()

## Appendix B：面试拓展（高频问答）

### 高频面试题

**Q1：GroupViT 为什么不需要像素级分割标注，也能做 zero-shot segmentation？**

- 训练目标不是像素交叉熵，而是图文对比学习；
- 图像编码器内部有 grouping 机制，会为了更好地和文本对齐，自动形成语义区域；
- 推理时再把 final groups 的类别回投到 patch / pixel；
- 因此 supervision 在“图文匹配”层面，分割能力是模型结构自然涌现出来的。

**Q2：Grouping Block 与普通 self-attention 的区别是什么？**

- self-attention 主要做 token 间信息交互；
- Grouping Block 明确执行“谁属于哪个 group”的分配；
- 它更像可学习聚类，而不仅仅是上下文混合；
- 这一步让 token 表示从规则网格走向不规则语义区域。

**Q3：为什么训练时要用 Gumbel-Softmax，而不是直接 argmax？**

- argmax 是离散操作，不可导；
- Gumbel-Softmax 可以近似 one-hot，同时保留梯度；
- 这样 grouping 的离散分配就能端到端训练；
- 推理时再切换成真正的硬分配即可。

**Q4：GroupViT 和 CLIP 的关系是什么？**

- 两者都用图文对比学习；
- CLIP 主要学习全局图像-文本对齐；
- GroupViT 在图像编码器里额外加入 group tokens 和 grouping block；
- 所以 GroupViT 比 CLIP 多了“区域级结构归纳偏置”，能做 zero-shot segmentation。

**Q5：为什么 GroupViT 的背景类通常更难？**

- 背景不是一个单一语义，而是很多弱语义区域的混合；
- 图文对数据更强调“显著主体”，对背景描述通常不足；
- 对比学习更容易把注意力集中到可命名目标上；
- 所以前景轮廓常常比背景类别判断更可靠。

**Q6：为什么最终只有 8 个 groups 会成为限制？**

- final groups 数量本质上决定了模型能显式表达多少个区域槽位；
- 如果一张图里目标非常多，group 槽位会不够；
- 槽位不足时，不同语义可能被迫合并；
- 因而最终 group 数是表达能力与效率之间的折中。

**Q7：如果让你改进 GroupViT，你会优先改哪里？**

- 增加或自适应调整 final group 数量；
- 改进背景建模与 open-vocabulary 推理策略；
- 引入多尺度 grouping，兼顾小目标和大区域；
- 加入跨图像一致性约束，让同类 group 的语义更稳定。

### 延伸阅读与对比

| 对比维度 | CLIP | LSeg | GroupViT | SAM |
|---|---|---|---|---|
| 监督信号 | 图文对 | 像素标注 + 文本特征 | 图文对 | 大规模 mask 数据 |
| 主要目标 | 图文检索 / 分类 | 开放词汇分割 | 文本监督分组 + zero-shot 分割 | Promptable segmentation |
| 是否内建 grouping | 否 | 否 | 是 | 否 |
| 是否直接输出类别语义 | 是 | 是 | 是 | 否（更偏区域提议） |
| 典型优势 | 全局语义强 | 分割类别对齐强 | 无像素标注也能分割 | 通用分割能力强 |

### 进阶探索方向

- **SegCLIP / CLIPSeg**：继续研究“文本监督如何更稳定地落到像素空间”；
- **SAM + 文本模型组合**：把高质量区域提议和语言语义对齐结合起来；
- **动态 group token**：让不同图像使用不同数量的区域槽位；
- **多尺度 GroupViT**：提升小目标与复杂背景场景下的鲁棒性。